# 03 - Ingénierie des Caractéristiques Multi-Sources

## Objectif
Concevoir une représentation unifiée des caractéristiques qui :
1. Préserve les informations significatives de chaque catastrophe
2. Permet l'apprentissage inter-catastrophes
3. Gère les données manquantes de manière appropriée (MNAR)
4. Intègre des données contextuelles externes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from src.data_processing import (
    load_titanic, load_lusitania, load_sewol,
    clean_titanic, clean_lusitania, clean_sewol,
    create_unified_dataset, save_processed_data
)
from src.feature_engineering import (
    load_external_data, add_ship_features, add_weather_features,
    add_economic_features, create_social_connectivity,
    impute_age_mice, analyze_mnar_patterns, create_age_groups,
    create_interaction_features, prepare_features_for_modeling,
    full_feature_engineering_pipeline
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)

print('Modules chargés.')

In [ ]:
# Charger et unifier les données
titanic_clean = clean_titanic(load_titanic())
lusitania_clean = clean_lusitania(load_lusitania())
sewol_clean = clean_sewol(load_sewol())
unified = create_unified_dataset(titanic_clean, lusitania_clean, sewol_clean)

print(f"Dataset unifié : {unified.shape}")
unified.head()

---
## 1. Encodage du Statut Social Unifié

### Problème
Les systèmes de classe diffèrent fondamentalement entre les époques :
- **Titanic (1912)** : Système de classes de billets (1ère, 2ème, 3ème) reflétant la hiérarchie sociale Edwardienne
- **Lusitania (1915)** : Similaire mais avec des départements d'équipage distincts
- **Sewol (2014)** : Ferry domestique coréen, pas de système de classes traditionnel

### Solution
Créer une échelle unifiée 1-3 basée sur le niveau d'accès aux ressources et l'autonomie d'évacuation :
- **1 (Haute)** : Accès prioritaire (1ère classe, officiers supérieurs)
- **2 (Moyenne)** : Accès normal (2ème classe, équipage)
- **3 (Basse)** : Accès limité (3ème classe, étudiants sans autonomie)

In [ ]:
# Analyse du statut social par catastrophe
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Statut Social Unifié par Catastrophe', fontsize=14, fontweight='bold')

for i, disaster in enumerate(['titanic', 'lusitania', 'sewol']):
    subset = unified[unified['disaster'] == disaster]
    survival_by_class = subset.groupby('social_class')['survived'].agg(['mean', 'count'])
    bars = axes[i].bar(survival_by_class.index, survival_by_class['mean'],
                       color=['gold', 'steelblue', 'coral'])
    axes[i].set_title(f'{disaster.capitalize()}')
    axes[i].set_xlabel('Classe sociale')
    axes[i].set_ylabel('Taux de survie')
    axes[i].set_xticks([1, 2, 3])
    axes[i].set_xticklabels(['Haute', 'Moyenne', 'Basse'])
    # Annoter avec le count
    for j, (idx, row) in enumerate(survival_by_class.iterrows()):
        axes[i].annotate(f'n={int(row["count"])}',
                        xy=(idx, row['mean']), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### Justification : Traitement de l'Équipage

Les membres d'équipage ne doivent **pas** être traités identiquement entre les catastrophes :

- **Titanic/Lusitania** : L'équipage avait une connaissance du navire et un accès aux canots, mais aussi le devoir de rester pour aider à l'évacuation. Leur survie dépendait du conflit entre accès et devoir.
- **Sewol** : L'équipage (capitaine en tête) a été parmi les premiers à quitter le navire, abandonnant les passagers. Leur taux de survie est disproportionnellement élevé.

→ Le flag `is_crew` capture cette distinction, et les interactions `crew_modern` permettent au modèle d'apprendre des comportements d'équipage différents selon l'époque.

In [ ]:
# Survie équipage vs passagers par catastrophe
crew_survival = unified.groupby(['disaster', 'is_crew'])['survived'].agg(['mean', 'count'])
print("Taux de survie : Équipage vs Passagers")
print(crew_survival)
print("\nObservation clé : L'équipage du Sewol a un taux de survie nettement supérieur")
print("aux passagers, contrairement au Titanic où c'est l'inverse.")

---
## 2. Gestion des Données Manquantes (MNAR)

### Pourquoi les simples indicatrices sont insuffisantes

Les données manquantes dans nos datasets ne sont **pas aléatoires** (MNAR) :

1. **Âge manquant dans le Titanic** : Corrélé avec la 3ème classe (les registres d'immigration étaient moins rigoureux pour les passagers les plus pauvres)
2. **Cabine manquante dans le Titanic** : 77% de valeurs manquantes, quasi-exclusivement en 3ème classe
3. **Âge manquant dans le Sewol** : Certains passagers "normaux" n'avaient pas de réservation formelle

Une simple indicatrice `age_is_missing=1` ne récupère pas l'information sur la valeur probable de l'âge et crée un proxy indirect de la classe sociale, ce qui introduit de la multicolinéarité avec `social_class`.

### Solution : MICE (Multiple Imputation by Chained Equations)
MICE utilise les corrélations entre toutes les variables pour estimer les valeurs manquantes de manière itérative.

In [ ]:
# Analyse MNAR avant imputation
print("=== Analyse MNAR : Âge manquant ===")
print(f"\nÂge manquant par catastrophe :")
print(unified.groupby('disaster')['age'].apply(lambda x: x.isna().mean()).round(4))

print(f"\nÂge manquant par classe sociale :")
print(unified.groupby('social_class')['age'].apply(lambda x: x.isna().mean()).round(4))

print(f"\nTaux de survie pour âge manquant vs non-manquant :")
unified['age_missing'] = unified['age'].isna().astype(int)
print(unified.groupby('age_missing')['survived'].mean().round(4))

In [ ]:
# Visualisation des patterns MNAR
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Patterns de Données Manquantes (MNAR)', fontsize=14, fontweight='bold')

# Par catastrophe
unified.groupby('disaster')['age'].apply(lambda x: x.isna().mean()).plot(
    kind='bar', ax=axes[0], color='coral')
axes[0].set_title('% Âge Manquant par Catastrophe')
axes[0].set_ylabel('Proportion manquante')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Par classe sociale
unified.groupby('social_class')['age'].apply(lambda x: x.isna().mean()).plot(
    kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('% Âge Manquant par Classe Sociale')
axes[1].set_ylabel('Proportion manquante')
axes[1].set_xticks([0, 1, 2])
axes[1].set_xticklabels(['Haute', 'Moyenne', 'Basse'], rotation=0)

# Survie vs missing
unified.groupby('age_missing')['survived'].mean().plot(
    kind='bar', ax=axes[2], color=['mediumseagreen', 'coral'])
axes[2].set_title('Taux de Survie : Âge Connu vs Manquant')
axes[2].set_xticklabels(['Âge connu', 'Âge manquant'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Appliquer MICE
print("Application de l'imputation MICE...")
df_imputed = impute_age_mice(unified)

# Comparer les distributions avant/après
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

unified['age'].dropna().hist(bins=30, ax=axes[0], alpha=0.7, label='Avant imputation',
                               color='steelblue', edgecolor='black')
axes[0].set_title('Distribution Âge Avant MICE')
axes[0].set_xlabel('Âge')
axes[0].legend()

df_imputed['age'].hist(bins=30, ax=axes[1], alpha=0.7, label='Après imputation',
                        color='mediumseagreen', edgecolor='black')
axes[1].set_title('Distribution Âge Après MICE')
axes[1].set_xlabel('Âge')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nValeurs manquantes avant : {unified['age'].isna().sum()}")
print(f"Valeurs manquantes après : {df_imputed['age'].isna().sum()}")

# Analyse MNAR post-imputation
mnar_results = analyze_mnar_patterns(df_imputed)
print(f"\nSurvie par age_was_missing : {mnar_results['survival_by_age_missing']}")
print(f"Classe par age_was_missing : {mnar_results['class_by_age_missing']}")

---
## 3. Connectivité Sociale

### Hypothèse
Les personnes voyageant en groupe ont un avantage de survie grâce à :
- L'entraide mutuelle pendant l'évacuation
- La prise de décision collective (quitter la cabine vs rester)
- La protection des enfants par les adultes du groupe

### Défi
Les informations familiales sont enregistrées différemment :
- Titanic : SibSp (frères/sœurs + conjoints) + Parch (parents + enfants)
- Lusitania : Notes textuelles sur les compagnons de voyage
- Sewol : Catégorie (famille, groupe, personnel, classe scolaire)

In [ ]:
# Appliquer la connectivité sociale
df_social = create_social_connectivity(df_imputed)

# Analyse de la connectivité sociale
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Connectivité Sociale et Survie', fontsize=14, fontweight='bold')

# Survie solo vs groupe
df_social.groupby(['disaster', 'is_alone'])['survived'].mean().unstack().plot(
    kind='bar', ax=axes[0], color=['mediumseagreen', 'coral'])
axes[0].set_title('Survie : Seul vs En Groupe')
axes[0].set_ylabel('Taux de survie')
axes[0].legend(['En groupe', 'Seul'])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Taille de groupe vs survie
group_survival = df_social.groupby('group_size')['survived'].agg(['mean', 'count'])
group_survival = group_survival[group_survival['count'] >= 5]  # Au moins 5 individus
axes[1].bar(group_survival.index, group_survival['mean'], color='steelblue')
axes[1].set_title('Taux de Survie par Taille de Groupe')
axes[1].set_xlabel('Taille du groupe')
axes[1].set_ylabel('Taux de survie')

# Score de connectivité sociale
for disaster in ['titanic', 'lusitania', 'sewol']:
    subset = df_social[df_social['disaster'] == disaster]
    axes[2].scatter(subset['social_network_score'], subset['survived'] + np.random.normal(0, 0.02, len(subset)),
                   alpha=0.3, label=disaster.capitalize(), s=10)
axes[2].set_title('Score de Connectivité Sociale vs Survie')
axes[2].set_xlabel('Score de connectivité sociale')
axes[2].set_ylabel('Survie (jittered)')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 4. Intégration des Données Contextuelles

In [ ]:
# Charger les données externes
ship_specs, weather, economics = load_external_data()

print("=== Spécifications des navires ===")
print(ship_specs[['disaster', 'tonnage_grt', 'total_capacity', 'lifeboat_capacity', 'sinking_duration_min', 'water_temp_celsius']].to_string(index=False))

print("\n=== Conditions météorologiques ===")
print(weather[['disaster', 'sea_state', 'visibility', 'wind_speed_knots', 'water_temp_celsius', 'daylight']].to_string(index=False))

print("\n=== Indicateurs économiques ===")
print(economics.to_string(index=False))

In [ ]:
# Appliquer le pipeline complet
df_enriched = add_ship_features(df_social, ship_specs)
df_enriched = add_weather_features(df_enriched, weather)
df_enriched = add_economic_features(df_enriched, economics)
df_enriched = create_age_groups(df_enriched)
df_enriched = create_interaction_features(df_enriched)

print(f"\nDataset enrichi : {df_enriched.shape}")
print(f"\nNouvelles colonnes ajoutées :")
new_cols = set(df_enriched.columns) - set(unified.columns)
for col in sorted(new_cols):
    print(f"  - {col}")

df_enriched.head()

In [ ]:
# Visualisation des features contextuelles
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Features Contextuelles par Catastrophe', fontsize=14, fontweight='bold')

# Température de l'eau
df_enriched.groupby('disaster')['water_temp_celsius'].first().plot(
    kind='bar', ax=axes[0, 0], color=['steelblue', 'coral', 'mediumseagreen'])
axes[0, 0].set_title('Température de l\'Eau (°C)')
axes[0, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Point de congélation')
axes[0, 0].legend()
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=0)

# Ratio canots de sauvetage
df_enriched.groupby('disaster')['lifeboat_ratio'].first().plot(
    kind='bar', ax=axes[0, 1], color=['steelblue', 'coral', 'mediumseagreen'])
axes[0, 1].set_title('Ratio Canots / Capacité')
axes[0, 1].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Ratio 1:1')
axes[0, 1].legend()
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)

# Durée du naufrage
df_enriched.groupby('disaster')['sinking_duration_min'].first().plot(
    kind='bar', ax=axes[1, 0], color=['steelblue', 'coral', 'mediumseagreen'])
axes[1, 0].set_title('Durée du Naufrage (minutes)')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)

# GDP par habitant
df_enriched.groupby('disaster')['gdp_per_capita_usd_2014'].first().plot(
    kind='bar', ax=axes[1, 1], color=['steelblue', 'coral', 'mediumseagreen'])
axes[1, 1].set_title('PIB par Habitant (USD 2014)')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

---
## 5. Matrice de Corrélation

In [ ]:
# Sélectionner les features numériques pour la corrélation
numeric_cols = df_enriched.select_dtypes(include=[np.number]).columns
corr_cols = [c for c in numeric_cols if c not in ['disaster_year', 'original_index']]

corr_matrix = df_enriched[corr_cols].corr()

# Heatmap
plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Matrice de Corrélation des Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top corrélations avec survived
print("\nTop corrélations avec 'survived' :")
surv_corr = corr_matrix['survived'].drop('survived').abs().sort_values(ascending=False)
for feat, val in surv_corr.head(10).items():
    direction = '+' if corr_matrix.loc['survived', feat] > 0 else '-'
    print(f"  {feat}: {direction}{val:.4f}")

---
## 6. Sauvegarde du Dataset Enrichi

In [ ]:
# Sauvegarder le dataset enrichi
save_processed_data(df_enriched, 'enriched_dataset.csv')

# Préparer les features pour la modélisation
X, y, scaler, feature_names = prepare_features_for_modeling(df_enriched, scale=True)

print(f"\nFeatures pour la modélisation : {len(feature_names)}")
for i, feat in enumerate(feature_names, 1):
    print(f"  {i:2d}. {feat}")

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y distribution: {y.value_counts().to_dict()}")

---
## 7. Résumé des Features Créées

| Catégorie | Feature | Description | Justification |
|-----------|---------|-------------|---------------|
| **Démographique** | age | Âge imputé par MICE | Variable fondamentale |
| | sex_binary | Genre (1=F, 0=M) | Facteur historique clé |
| | is_child | Moins de 18 ans | "Children first" |
| | age_group | Catégorie d'âge | Non-linéarité de l'effet de l'âge |
| **Social** | social_class | Classe unifiée 1-3 | Accès aux ressources |
| | is_crew | Membre d'équipage | Rôle différent dans l'évacuation |
| | group_size | Taille du groupe | Entraide |
| | is_alone | Voyage seul | Isolement |
| | group_advantage | Score de groupe | Rendements décroissants |
| | social_network_score | Connectivité composite | Score multi-facteurs |
| **Interaction** | female_high_class | Femme + haute classe | Intersection privilège |
| | male_low_class | Homme + basse classe | Intersection vulnérabilité |
| | child_in_group | Enfant en groupe | Protection familiale |
| | crew_modern | Équipage + post-1950 | Changement de comportement |
| **Contextuel** | water_temp_celsius | Température eau | Hypothermie |
| | lifeboat_ratio | Ratio canots/capacité | Chances d'évacuation |
| | sinking_duration_min | Durée du naufrage | Temps d'évacuation |
| | sinking_speed | Vitesse de naufrage | Urgence |
| | wind_speed_knots | Vent | Conditions de sauvetage |
| | wave_height_m | Hauteur des vagues | Difficulté d'évacuation |
| | daylight_binary | Jour/nuit | Visibilité |
| | gdp_per_capita_usd_2014 | PIB par habitant | Contexte économique |
| **MNAR** | age_was_missing | Âge était manquant | Proxy de statut social |